In [1]:
import os
os.chdir("../..")

In [2]:
import torch
from models.mlrod import MLROD_model
from models.RamanFormer import RamanFormer
from models.RamanNet import RamanNet
from models.SANet import ScaleAdaptiveNet
from models.transformer import ViT
from torch.utils.data import Dataset,DataLoader
import pickle
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
from datasets.Bacteria_ID.config import ATCC_GROUPINGS
import numpy as np

In [3]:
class Bacteria_Dataset(Dataset):
    def __init__(self,X_path,y_path,num_classes=30):
        """
        X_path is a string containing the path to the spectra
        y_path is a string containing the path to the labels for the spectra
        num_classes is an integer corresponding to an 8 class or 30 class problem 
        """
        super().__init__()
        self.X = np.load(X_path)
        self.y = np.load(y_path)
        self.num_classes = num_classes
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self,index):
        data = torch.Tensor(self.X[index]).unsqueeze(0) #of shape (1,1000)
        data = data/(data.max())
        label = self.y[index]
        if self.num_classes==8:
            label = ATCC_GROUPINGS[label]
        return data,int(label)

In [4]:
def test_f1(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [5]:
def test_f1_RamanNet(model,device,test_dataloader):
    model.eval()
    loop = tqdm(test_dataloader)

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for i, (X, y) in enumerate(loop):
            X = X.to(device)
            y = y.to(device)

            y_pred,_ = model(X)#of shape (b,num_classes)                  
            preds = y_pred.argmax(dim=1) #of shape (b)

            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy() #of shape (num_samples,)
    all_targets = torch.cat(all_targets).numpy() #of shape (num_samples,)

    accuracy = 100.0 * (all_preds == all_targets).mean()
    precision = precision_score(all_targets, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_targets, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return accuracy, precision, recall, f1

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [7]:
test_set = Bacteria_Dataset("datasets/Bacteria_ID/X_test.npy","datasets/Bacteria_ID/y_test.npy",8)
test_loader = DataLoader(test_set, batch_size=32, num_workers=8,shuffle=False)
print(f"The number of elements in the test set is {len(test_loader.dataset)}")

The number of elements in the test set is 3000


In [8]:
Bacteria_mlrod = MLROD_model(num_classes=8,sp_size=1000).to(device)
Bacteria_mlrod.load_state_dict(torch.load("results/trained_models/Bacteria_eight_mlrod_18_99.0_.pt"))

Bacteria_RamanFormer = RamanFormer(num_classes=8,sp_size=1000,patch_size=125).to(device)
Bacteria_RamanFormer.load_state_dict(torch.load("results/trained_models/Bacteria_eight_RamanFormer_36_99.17_.pt"))

Bacteria_RamanNet = RamanNet(num_classes=8,sp_size=1000).to(device)
Bacteria_RamanNet.load_state_dict(torch.load("results/trained_models/Bacteria_eight_RamanNet_34_96.83_.pt"))

Bacteria_SANet = ScaleAdaptiveNet(num_classes=8).to(device)
Bacteria_SANet.load_state_dict(torch.load("results/trained_models/Bacteria_eight_SANet_39_98.5_.pt"))

Bacteria_transformer = ViT(patch_size=125,p=0.1,num_classes=8).to(device)
Bacteria_transformer.load_state_dict(torch.load("results/trained_models/Bacteria_eight_transformer_27_98.0_.pt"))

<All keys matched successfully>

In [9]:
all_acc, _, _, all_f1 = test_f1(Bacteria_mlrod,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_15216/1026935260.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 94/94 [00:02<00:00, 31.39it/s]

97.63\% & 0.9804 \\


In [10]:
all_acc, _, _, all_f1 = test_f1(Bacteria_SANet,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_15216/1930381594.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 94/94 [00:01<00:00, 55.32it/s] 

96.77\% & 0.9699 \\


In [11]:
all_acc, _, _, all_f1 = test_f1_RamanNet(Bacteria_RamanNet,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_15216/3148680625.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 94/94 [00:01<00:00, 63.94it/s] 

96.87\% & 0.9711 \\


In [12]:
all_acc, _, _, all_f1 = test_f1(Bacteria_transformer,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_15216/3539688219.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 94/94 [00:01<00:00, 48.73it/s] 

96.63\% & 0.9676 \\


In [13]:
all_acc, _, _, all_f1 = test_f1(Bacteria_RamanFormer,device,test_loader)
print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_15216/1676594397.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(f"{round(all_acc,2)}\% & {round(all_f1,4)} \\\\")
100%|██████████| 94/94 [00:01<00:00, 72.29it/s] 

97.1\% & 0.9752 \\
